In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import scipy.integrate as integrate
from scipy.optimize import curve_fit

mol_frac = np.array([0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1])
ri = np.array([1.37250, 1.37000, 1.36750, 1.36450, 1.35800,
               1.35500, 1.35000, 1.34600, 1.33850, 1.33450, 1.32350])
ri_err = 1e-3
ri_errs = np.full_like(ri, ri_err)

# Polynomial function
def ri_FIND(x, a, b, c):
    return a * x**2 + b * x + c

# Curve fit with uncertainties
params, cov = curve_fit(ri_FIND, mol_frac, ri, sigma=ri_errs, absolute_sigma=True)

a, b, c = params

def mf_FIND(ri_val, ri_err=ri_err):
    # Solve quadratic: ax^2 + bx + (c - ri_val) = 0
    coeffs = [a, b, c - ri_val]
    roots = np.roots(coeffs)
    
    # Select valid roots
    valid_roots = roots[np.isreal(roots) & (roots >= 0) & (roots <= 1)].real
    if len(valid_roots) == 0:
        return np.nan, np.nan
    x = valid_roots[0]  # or handle multiple roots as discussed
    
    # Error from parameter uncertainties (optional but recommended)
    # Partial derivatives of x with respect to a, b, c
    sqrt_part = np.sqrt(b**2 - 4*a*(c - ri_val))
    dx_da = (b - sqrt_part)/(2*a)**2 + (c - ri_val)/(a*sqrt_part)
    dx_db = (-1 + b/sqrt_part)/(2*a)
    dx_dc = 1/sqrt_part
    
    # Variance from parameters
    var_x_params = (dx_da**2 * cov[0,0] + 
                   dx_db**2 * cov[1,1] + 
                   dx_dc**2 * cov[2,2] +
                   2*dx_da*dx_db*cov[0,1] +
                   2*dx_da*dx_dc*cov[0,2] +
                   2*dx_db*dx_dc*cov[1,2])
    
    # Total error
    total_error = var_x_params
    
    return x, total_error

## -----------------------------------
    
def VLE_solve(x): 
  T = 72
  G12 = 1.689
  G21 = 0.5445
  x2 = 1-x
  R1 = x + x2*G12
  R2 = x2 + x*G21

  P1_MeOH = 0.00131579 * (10**(7.87863 - 1473.11 / (T + 230.0)))
  P2_IPA = 0.00131579 * (10**(6.66040 - 813.055 / (T + 132.93)))

  gamma1 = math.exp(-math.log(R1) + x2*((G12/R1) - (G21/R2)))
  gamma2 = math.exp(-math.log(R2) - x*((G12/R1) - (G21/R2)))

  P_total = x * gamma1 * P1_MeOH + x2 * gamma2 * P2_IPA

  y = x*(P1_MeOH/P_total)*gamma1
  return y


In [2]:
import math

def weighted_avg(xMeOH):
    xIPA = 1 - xMeOH
    mass_density = (xMeOH*0.792) + (xIPA*0.786)
    return mass_density
    
def calculate_sigma_f(x, sigma_x, m, sigma_m, t):    
            ###Constants
    rho_A = 0.792    # density of MeOH in g/mL
    rho_B = 0.786     # density of IPA in g/mL
    rho_eff = weighted_avg(x) # effective density of the distillate
    t = t/60 ## minute
    sigma_t = 5
    
            ### Error Prop
    term1 = (sigma_m / (rho_eff * t)) ** 2
    term2 = ((m * (rho_A - rho_B) * sigma_x) / (rho_eff ** 2 * t)) ** 2
    term3 = ((m * sigma_t) / (rho_eff * t ** 2)) ** 2
    sigma_f = math.sqrt(term1 + term2 + term3)
    
    FR = m/(t*rho_eff) ## in mL/min

    return FR, sigma_f

t1 = [0.99, 0.08, 7.99, 0.01, 2340.00]
t2 = [0.98, 0.08, 8.27, 0.01, 2301.00]
t3 = [0.94, 0.07, 8.27, 0.01, 2511.00]
t4 = [0.91, 0.07, 7.03, 0.01, 3160.00]

values = [t1, t2, t3, t4]

for give_list in values:
    FR, sigma_f = calculate_sigma_f(*give_list)
    print(f"{FR:.2f} {sigma_f:.2f} mL/min")

0.26 0.03 mL/min
0.27 0.04 mL/min
0.25 0.03 mL/min
0.17 0.02 mL/min


In [3]:
def calculate_sigma_f(t):    
    V = 50 ## mL
    sigma_V = 0.5 ## mL
    sigma_t = 1/60 ## 1 s

    FR = V/t

    term1 = sigma_V/t
    term2 = (V*sigma_t)/t**2

    sigma_f = math.sqrt(term1**2 + term2**2)
    
    return FR, sigma_f

values = [6.933, 8.183, 12.267]

for t in values:
    FR, sigma_f = calculate_sigma_f(t)
    print(f"{FR:.2f} {sigma_f:.2f}")

7.21 0.07
6.11 0.06
4.08 0.04
